# Intersectional Bias Analysis — Mistral (With Stigma)

This notebook analyzes bias in Mistral's responses across single and intersectional stigma prompts, and tests whether stigma cluster type predicts the intersectional interaction effect.

## 1. Setup

In [219]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data")

df_granite = pd.read_csv(DATA / "results__granite__with_single__original_positive_doubt_base.csv")
df_mistral = pd.read_csv(DATA / "results__mistral__with_single__original_positive_doubt_base.csv")


# Correct: biased = 1 when model gave the biased answer, not just when it said "yes"
df_granite["biased"] = (df_granite["model_answer"] == df_granite["biased_answer"]).astype(int)
df_mistral["biased"] = (df_mistral["model_answer"] == df_mistral["biased_answer"]).astype(int)
#df_llama["tbd"]



## 2. Data Quality

In [ ]:
valid = df[df["model_answer"].isin(["yes", "no"])].copy()
improper = df[~df["model_answer"].isin(["yes", "no"])]

print(f"Total rows:      {len(df):,}")
print(f"Valid (yes/no):  {len(valid):,}  ({100*len(valid)/len(df):.1f}%)")
print(f"Improper output: {len(improper):,}  ({100*len(improper)/len(df):.1f}%)")

## 3. Overall Bias Rate

In [ ]:
print(f"Overall bias rate: {valid['biased'].mean():.3f}  ({valid['biased'].mean()*100:.1f}%)")

In [ ]:
# Bias rate by prompt style
valid.groupby("prompt_style")["biased"].agg(["mean", "count"]).rename(columns={"mean": "bias_rate"}).round(3)

## 4. Single vs Intersectional Bias

In [ ]:
valid["intersectional"] = valid["stigma2"].notna()
valid.groupby("intersectional")["biased"].agg(["mean", "count"]).rename(
    columns={"mean": "bias_rate"},
    index={False: "single", True: "intersectional"}
).round(3)

s## 5. Interaction Effect

$$\text{Interaction Effect} = P(\text{bias} \mid A+B) - \frac{P(\text{bias} \mid A) + P(\text{bias} \mid B)}{2}$$

- **> 0**: intersection amplifies bias
- **< 0**: intersection dampens bias
- **= 0**: additive, no interaction

In [ ]:
# P(bias | stigma) for single-stigma rows
single_bias = (
    valid[valid["stigma2"].isna()]
    .groupby("stigma1")["biased"]
    .mean()
    .rename("single_bias_rate")
)

In [ ]:
# P(bias | A+B) — collapse (A,B) and (B,A) into one unordered pair by averaging
raw = (
    valid[valid["stigma2"].notna()]
    .groupby(["stigma1", "stigma2"])["biased"]
    .mean()
    .reset_index()
    .rename(columns={"biased": "combined_bias_rate"})
)

# Normalise pair order so (A,B) and (B,A) share the same key
raw["pair"] = raw.apply(lambda r: tuple(sorted([r["stigma1"], r["stigma2"]])), axis=1)
intersectional_bias = (
    raw.groupby("pair")["combined_bias_rate"]
    .mean()
    .reset_index()
)

intersectional_bias[["stigma1", "stigma2"]] = pd.DataFrame(
    intersectional_bias["pair"].tolist(), index=intersectional_bias.index
)
intersectional_bias = intersectional_bias.drop(columns="pair")

intersectional_bias = intersectional_bias.merge(single_bias.rename("bias_A"), left_on="stigma1", right_index=True, how="left")

intersectional_bias = intersectional_bias.merge(single_bias.rename("bias_B"), left_on="stigma2", right_index=True, how="left")

intersectional_bias["interaction_effect"] = (
    intersectional_bias["combined_bias_rate"] - (intersectional_bias["bias_A"] + intersectional_bias["bias_B"]) / 2
)

intersectional_bias = intersectional_bias.round(3).sort_values("interaction_effect", ascending=False)
intersectional_bias

## 6. Cluster Analysis

Joins Pachankis et al. cluster labels to see whether stigma type predicts the interaction effect.

In [ ]:
clusters = pd.read_csv(DATA / "templates/neostigmas.csv", usecols=["Stigma", "Cluster"])

ia = intersectional_bias.merge(clusters.rename(columns={"Stigma": "stigma1", "Cluster": "cluster_A"}), on="stigma1", how="left")
ia = ia.merge(clusters.rename(columns={"Stigma": "stigma2", "Cluster": "cluster_B"}), on="stigma2", how="left")

ia["cluster_pair"] = ia.apply(
    lambda r: " × ".join(sorted([r["cluster_A"], r["cluster_B"]])), axis=1
)

ia[["stigma1", "stigma2", "cluster_A", "cluster_B", "cluster_pair", "combined_bias_rate", "bias_A", "bias_B", "interaction_effect"]].head(10)

In [ ]:
# Mean interaction effect by cluster pair
cluster_pair_summary = (
    ia.groupby("cluster_pair")["interaction_effect"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_interaction", "std": "std_interaction"})
    .sort_values("mean_interaction", ascending=False)
    .round(3)
)
cluster_pair_summary

In [ ]:
# Mean interaction effect by individual cluster
cluster_A_summary = ia.groupby("cluster_A")["interaction_effect"].mean().rename("mean_interaction_as_A")
cluster_B_summary = ia.groupby("cluster_B")["interaction_effect"].mean().rename("mean_interaction_as_B")

cluster_view = pd.concat([cluster_A_summary, cluster_B_summary], axis=1)
cluster_view["mean_overall"] = cluster_view.mean(axis=1)
cluster_view.sort_values("mean_overall", ascending=False).round(3)

## 7. Bootstrap Confidence Intervals

5,000 bootstrap resamples of stigma pairs per cluster group. `excludes_zero = True` indicates the effect is reliably non-zero at 95% confidence.

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 5000

def bootstrap_ci(data, n_boot=N_BOOT, ci=95):
    data = np.array(data)
    boot_means = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return data.mean(), lower, upper

# By cluster pair
boot_rows = []
for pair, group in ia.groupby("cluster_pair"):
    mean, lower, upper = bootstrap_ci(group["interaction_effect"].values)
    boot_rows.append({
        "cluster_pair":     pair,
        "mean_interaction": round(mean, 4),
        "ci_lower":         round(lower, 4),
        "ci_upper":         round(upper, 4),
        "n_stigma_pairs":   len(group),
        "excludes_zero":    lower > 0 or upper < 0,
    })

boot_pairs = pd.DataFrame(boot_rows).sort_values("mean_interaction", ascending=False)
boot_pairs

In [ ]:
# By individual cluster
boot_cluster_rows = []
for cluster, group in ia.groupby("cluster_A"):
    mean, lower, upper = bootstrap_ci(group["interaction_effect"].values)
    boot_cluster_rows.append({
        "cluster":          cluster,
        "mean_interaction": round(mean, 4),
        "ci_lower":         round(lower, 4),
        "ci_upper":         round(upper, 4),
        "n_pairs":          len(group),
        "excludes_zero":    lower > 0 or upper < 0,
    })

boot_clusters = pd.DataFrame(boot_cluster_rows).sort_values("mean_interaction", ascending=False)
boot_clusters

## 8. Feature Regression

Uses the Pachankis et al. dimension scores (Visibility, Persistent Course, Disrupt, Unappealing Aesthetics, Controllable Origin, Peril) to predict the interaction effect. Each pair feature is the mean of stigma A and stigma B scores for that dimension.

In [ ]:
FEATURE_COLS = ["Visibility", "Persistent Course", "Disrupt",
                "Unappealing Aesthetics", "Controllable Origin", "Peril"]

# Parse "3.68 / 20" → 3.68
features = pd.read_csv(DATA / "templates/neostigmas.csv",
                       usecols=["Stigma"] + FEATURE_COLS)
for col in FEATURE_COLS:
    features[col] = features[col].astype(str).str.extract(r"^([\d.]+)").astype(float)

# Merge features for stigma1 and stigma2
ia_feat = intersectional_bias.merge(
    features.rename(columns={"Stigma": "stigma1"}).add_suffix("_A").rename(columns={"stigma1_A": "stigma1"}),
    on="stigma1", how="left"
)
ia_feat = ia_feat.merge(
    features.rename(columns={"Stigma": "stigma2"}).add_suffix("_B").rename(columns={"stigma2_B": "stigma2"}),
    on="stigma2", how="left"
)

# Combined pair feature = mean of A and B per dimension
for col in FEATURE_COLS:
    ia_feat[col] = (ia_feat[f"{col}_A"] + ia_feat[f"{col}_B"]) / 2

ia_feat = ia_feat.dropna(subset=FEATURE_COLS + ["interaction_effect"])

# ── Correlations ──────────────────────────────────────────────────────────────
corr = ia_feat[FEATURE_COLS + ["interaction_effect"]].corr()["interaction_effect"].drop("interaction_effect")
print("Pearson correlation with interaction_effect:")
print(corr.sort_values().round(4).to_string())

# ── OLS Regression ────────────────────────────────────────────────────────────
import statsmodels.api as sm

X = sm.add_constant(ia_feat[FEATURE_COLS])
y = ia_feat["interaction_effect"]
ols = sm.OLS(y, X).fit()
print("\n", ols.summary())

## 9. Deep Dive — Threatening × Sociodemographic Pairs

In [ ]:
# Filter intersectional_bias to only Threatening × Sociodemographic pairs
threat_socio = ia[
    ia["cluster_pair"] == "2 - Threatening × 3 - Sociodemographic"
][["stigma1", "stigma2", "cluster_A", "cluster_B",
   "combined_bias_rate", "bias_A", "bias_B", "interaction_effect"]].copy()

threat_socio = threat_socio.sort_values("interaction_effect")
print(f"Total pairs in Threatening × Sociodemographic: {len(threat_socio)}")
print(f"Mean interaction effect: {threat_socio['interaction_effect'].mean():.4f}")
print()

# Ensure Threatening stigma is always in one column for readability
def reorder(row):
    if row["cluster_A"] == "2 - Threatening":
        return row["stigma1"], row["stigma2"], row["bias_A"], row["bias_B"]
    return row["stigma2"], row["stigma1"], row["bias_B"], row["bias_A"]

threat_socio[["threatening_stigma", "sociodemographic_stigma", "bias_threatening", "bias_sociodemographic"]] =     threat_socio.apply(reorder, axis=1, result_type="expand")

threat_socio = threat_socio[[
    "threatening_stigma", "sociodemographic_stigma",
    "bias_threatening", "bias_sociodemographic",
    "combined_bias_rate", "interaction_effect"
]].sort_values("interaction_effect", ascending=False)

threat_socio.round(3)

In [ ]:
# All pairs involving any Threatening stigma
threatening_all = ia[
    (ia["cluster_A"] == "2 - Threatening") | (ia["cluster_B"] == "2 - Threatening")
].copy()

# Normalize so threatening stigma is always in the first column
def reorder_threatening(row):
    if row["cluster_A"] == "2 - Threatening":
        return row["stigma1"], row["stigma2"], row["cluster_B"], row["bias_A"], row["bias_B"]
    return row["stigma2"], row["stigma1"], row["cluster_A"], row["bias_B"], row["bias_A"]

threatening_all[["threatening_stigma", "partner_stigma", "partner_cluster", "bias_threatening", "bias_partner"]] = \
    threatening_all.apply(reorder_threatening, axis=1, result_type="expand")

threatening_table = threatening_all[[
    "threatening_stigma", "partner_stigma", "partner_cluster",
    "bias_threatening", "bias_partner", "combined_bias_rate", "interaction_effect"
]].sort_values("interaction_effect", ascending=False).round(3)

print(f"Total pairs involving a Threatening stigma: {len(threatening_table)}")
print(f"Mean interaction effect: {threatening_table['interaction_effect'].mean():.4f}")
threatening_table

In [ ]:
single_bias.sort_values(ascending=False).round(3).to_frame()

In [ ]:
single_bias.sort_values(ascending=False).round(3).to_frame().to_csv(DATA / "single_stigma_bias_rates.csv")
print("Saved to data/single_stigma_bias_rates.csv")

## 10. Deep Dive — Intersex

In [ ]:
# All rows where Intersex appears (single or intersectional)
intersex = valid[
    (valid["stigma1"] == "Intersex") | (valid["stigma2"].fillna("") == "Intersex")
].copy()

print(f"Total Intersex rows: {len(intersex)}")
print(f"Overall bias rate:   {intersex['biased'].mean():.3f}")
print()

# ── Single stigma ──────────────────────────────────────────────────────────────
single = intersex[intersex["stigma2"].isna()]
print("Single stigma bias rate by prompt style:")
print(single.groupby("prompt_style")["biased"].mean().round(3).to_string())
print()

# ── Intersectional — paired with what? ────────────────────────────────────────
paired = intersex[intersex["stigma2"].notna()].copy()
paired["partner"] = paired.apply(
    lambda r: r["stigma2"] if r["stigma1"] == "Intersex" else r["stigma1"], axis=1
)

partner_bias = (
    paired.groupby("partner")["biased"]
    .mean()
    .rename("combined_bias_rate")
    .to_frame()
    .assign(single_bias_intersex=single_bias.get("Intersex", float("nan")))
)
partner_bias["interaction_effect"] = (
    partner_bias["combined_bias_rate"]
    - (partner_bias["single_bias_intersex"] + partner_bias["partner"].map(single_bias)) / 2
)

print("Intersex paired — sorted by interaction effect:")
partner_bias.sort_values("interaction_effect", ascending=False).round(3)

In [ ]:
overall_bias = (
    valid
    .groupby("stigma1")["biased"]
    .mean()
    .rename("overall_bias_rate")
)

stigma_table = pd.concat([single_bias, overall_bias], axis=1)
stigma_table = stigma_table.merge(
    clusters.rename(columns={"Stigma": "stigma1"}), on="stigma1", how="left"
).set_index("stigma1")

stigma_table.round(3).sort_values("single_bias_rate", ascending=False)

In [ ]:
stigma_table.round(3).sort_values("single_bias_rate", ascending=False).to_csv(DATA / "stigma_bias_rates.csv")
print("Saved to data/stigma_bias_rates.csv")

## 11. Statistical Testing of the Interaction Effect

### 11a. Wilcoxon Signed-Rank Test
Tests whether the distribution of interaction effects across all pairs is centered at zero. Non-parametric — no normality assumption.

### 11b. Sign Test (Binomial)
If intersectionality had no systematic effect, 50% of pairs should amplify and 50% should dampen. Tests whether the observed split differs significantly from 50/50.

In [ ]:
from scipy import stats

ie_values = intersectional_bias["interaction_effect"].dropna().values

stat, p = stats.wilcoxon(ie_values)
print("Wilcoxon Signed-Rank Test")
print(f"  H0: median interaction effect = 0")
print(f"  Statistic: {stat:.2f}")
print(f"  p-value:   {p:.2e}")
print(f"  Median IE: {ie_values.mean():.4f}")
result = 'Reject H0 — effect is reliably non-zero' if p < 0.05 else 'Fail to reject H0'
print(f"  Result:    {result}")

In [ ]:
ie = intersectional_bias["interaction_effect"].dropna()

n_pairs      = len(ie)
n_dampening  = (ie < 0).sum()
n_amplifying = (ie > 0).sum()
n_zero       = (ie == 0).sum()

# Binomial test: under H0 each pair is equally likely to amplify or dampen
binom = stats.binomtest(n_dampening, n=n_pairs - n_zero, p=0.5, alternative='greater')

print("Sign Test (Binomial)")
print(f"  Total pairs:      {n_pairs}")
print(f"  Dampening (IE<0): {n_dampening}  ({100*n_dampening/n_pairs:.1f}%)")
print(f"  Amplifying (IE>0):{n_amplifying}  ({100*n_amplifying/n_pairs:.1f}%)")
print(f"  Neutral (IE=0):   {n_zero}")
print(f"  p-value:          {binom.pvalue:.2e}")
result = 'Reject H0 — dampening is significantly more common than amplifying' if binom.pvalue < 0.05 else 'Fail to reject H0'
print(f"  Result:           {result}")

## 12. Order Effect — McNemar's Test

For each stigma pair, the same prompts exist in both orders: (A, B) and (B, A). McNemar's test checks whether the model gives different binary answers depending on which stigma appears first.

**H₀:** P(biased | A∩B) = P(biased | B∩A) — order does not matter

Rows are matched positionally within each (pair, prompt_style) group — both orderings were generated from the same pattern list in the same order.

- If no significant order effect → pool (A,B) and (B,A) for more power
- If significant → order itself is a finding: the model anchors on whichever stigma appears first

In [ ]:
from scipy.stats import chi2

inter = valid[valid["stigma2"].notna()].copy() # single stigma when stigma 2 == nan
inter["pair"] = inter.apply(lambda r: tuple(sorted([r["stigma1"], r["stigma2"]])), axis=1)
inter["is_ab"] = inter.apply(lambda r: r["stigma1"] < r["stigma2"], axis=1)

b_total = c_total = 0
skipped = 0

for (pair, style), group in inter.groupby(["pair", "prompt_style"]):
    ab = group[group["is_ab"]].sort_index()["biased"].values
    ba = group[~group["is_ab"]].sort_index()["biased"].values
    if len(ab) != len(ba) or len(ab) == 0:
        skipped += 1
        continue
    b_total += ((ab == 1) & (ba == 0)).sum()  # AB biased, BA not
    c_total += ((ab == 0) & (ba == 1)).sum()  # AB not, BA biased

# McNemar's statistic (with continuity correction)
mcnemar_stat = (abs(b_total - c_total) - 1) ** 2 / (b_total + c_total)
p_mcnemar = chi2.sf(mcnemar_stat, df=1)

print("McNemar's Test — Order Effect")
print(f"  Matched prompt pairs:      {b_total + c_total + 0:,} discordant")
print(f"  AB biased, BA not (b):     {b_total:,}")
print(f"  BA biased, AB not (c):     {c_total:,}")
print(f"  Skipped (unmatched pairs): {skipped}")
print(f"  McNemar statistic:         {mcnemar_stat:.4f}")
print(f"  p-value:                   {p_mcnemar:.2e}")
result = 'Reject H0 — prompt order significantly affects bias' if p_mcnemar < 0.05 else 'Fail to reject H0 — order does not significantly affect bias; pooling (A,B) and (B,A) is justified'
print(f"  Result: {result}")

In [ ]:
pair_results = []

for pair, group in inter.groupby("pair"):
    ab = group[group["is_ab"]].sort_index()["biased"].values
    ba = group[~group["is_ab"]].sort_index()["biased"].values
    if len(ab) != len(ba) or len(ab) == 0:
        continue
    b = ((ab == 1) & (ba == 0)).sum()
    c = ((ab == 0) & (ba == 1)).sum()
    if b + c == 0:
        continue
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    p = chi2.sf(stat, df=1)
    pair_results.append({
        "stigma_A": pair[0],
        "stigma_B": pair[1],
        "b (AB biased, BA not)": b,
        "c (BA biased, AB not)": c,
        "p_value": round(p, 4),
        "significant": p < 0.05,
        "direction": "A first → more bias" if b > c else "B first → more bias",
    })

pair_order_df = pd.DataFrame(pair_results)
sig = pair_order_df[pair_order_df["significant"]].sort_values("p_value")
print(f"Pairs with significant order effect: {len(sig)} / {len(pair_order_df)}  ({100*len(sig)/len(pair_order_df):.1f}%)")
sig

In [ ]:
pair_order = inter.groupby(["pair", "is_ab"])["biased"].mean().unstack()
pair_order.columns = ["BA", "AB"]
pair_order["order_diff"] = pair_order["AB"] - pair_order["BA"]
pair_order.sort_values("order_diff", ascending=False)

In [ ]:
inter["s1_is_drug"] = inter["stigma1"] == "Drug Dependency Remitted"
inter["s2_is_drug"] = inter["stigma2"] == "Drug Dependency Remitted"

# When drug is first vs second
drug_first = inter[inter["is_ab"] & inter["s1_is_drug"]]["biased"].mean()
drug_second = inter[~inter["is_ab"] & inter["s2_is_drug"]]["biased"].mean()
print(f"Drug as first stigma:  {drug_first:.4f}")
print(f"Drug as second stigma: {drug_second:.4f}")

In [ ]:
# Filter to pairs involving any Drug Dependency stigma
drug_mask = (
    pair_order_df["stigma_A"].str.contains("Drug", case=False) |
    pair_order_df["stigma_B"].str.contains("Drug", case=False)
)
drug_pairs = pair_order_df[drug_mask].copy()

# Identify the partner stigma (the non-drug one)
drug_pairs["drug_stigma"] = drug_pairs.apply(
    lambda r: r["stigma_A"] if "drug" in r["stigma_A"].lower() else r["stigma_B"], axis=1
)
drug_pairs["partner_stigma"] = drug_pairs.apply(
    lambda r: r["stigma_B"] if "drug" in r["stigma_A"].lower() else r["stigma_A"], axis=1
)

# Merge cluster of partner stigma
drug_pairs = drug_pairs.merge(
    clusters.rename(columns={"Stigma": "partner_stigma"}),
    on="partner_stigma", how="left"
)

# Summary by cluster
cluster_order_summary = (
    drug_pairs.groupby("Cluster")
    .agg(
        n_pairs=("p_value", "count"),
        n_significant=("significant", "sum"),
        mean_b=("b (AB biased, BA not)", "mean"),
        mean_c=("c (BA biased, AB not)", "mean"),
        mean_p=("p_value", "mean"),
    )
    .assign(pct_significant=lambda x: (100 * x["n_significant"] / x["n_pairs"]).round(1))
    .sort_values("pct_significant", ascending=False)
    .round(3)
)

print(f"Drug Dependency pairs: {len(drug_pairs)} total, {drug_pairs['significant'].sum()} significant")
print()
cluster_order_summary

## 13. Statistical Significance of Single Stigma Bias Rates

For each stigma, tests whether its bias rate is significantly higher than the base rate — the model's response rate with no stigma mentioned. Uses a one-sided binomial test per stigma.

In [ ]:
# Base rate — prompts with no stigma mentioned (prompt_style == 'base', single rows)
base_rate = valid[
    (valid["stigma2"].isna()) & (valid["prompt_style"] == "base")
]["biased"].mean()
print(f"Base rate (no stigma): {base_rate:.4f}")
print()

# Binomial test per stigma (excluding base style — only original/positive/doubt)
single_rows = valid[
    (valid["stigma2"].isna()) & (valid["prompt_style"] != "base")
]

sig_rows = []
for stigma, group in single_rows.groupby("stigma1"):
    n = len(group)
    k = group["biased"].sum()
    result = stats.binomtest(k, n=n, p=base_rate, alternative="greater")
    sig_rows.append({
        "stigma": stigma,
        "n": n,
        "bias_rate": round(k / n, 3),
        "base_rate": round(base_rate, 3),
        "p_value": round(result.pvalue, 4),
        "significant": result.pvalue < 0.05,
    })

single_sig_df = pd.DataFrame(sig_rows).sort_values("bias_rate", ascending=False)
print(f"Significant stigmas (p < 0.05): {single_sig_df['significant'].sum()} / {len(single_sig_df)}")
single_sig_df

## 14. Statistical Significance of Intersectional Pair Bias Rates

For each stigma pair, tests whether the combined bias rate is significantly higher than the base rate. Uses a one-sided binomial test — mirrors section 13 but for pairs rather than single stigmas.

In [ ]:
pair_rows = valid[
    (valid["stigma2"].notna()) & (valid["prompt_style"] != "base")
]

pair_sig_rows = []
for (s1, s2), group in pair_rows.groupby(["stigma1", "stigma2"]):
    n = len(group)
    k = group["biased"].sum()
    result = stats.binomtest(k, n=n, p=base_rate, alternative="greater")
    pair_sig_rows.append({
        "stigma1": s1,
        "stigma2": s2,
        "n": n,
        "bias_rate": round(k / n, 3),
        "base_rate": round(base_rate, 3),
        "p_value": round(result.pvalue, 4),
        "significant": result.pvalue < 0.05,
    })

pair_sig_df = pd.DataFrame(pair_sig_rows).sort_values("bias_rate", ascending=False)
print(f"Significant pairs (p < 0.05): {pair_sig_df['significant'].sum()} / {len(pair_sig_df)}  ({100*pair_sig_df['significant'].mean():.1f}%)")
print()
pair_sig_df

## 15. Kai Label Analysis

Replicates the cluster analysis (sections 6 & 7) using custom Kai labels instead of Pachankis clusters. Note: `reglion` and `religion` appear to be the same category — verify before interpreting.

In [ ]:
kai = pd.read_csv(DATA / "templates/kai_labels.csv")
kai["Kai"] = kai["Kai"].str.strip()   # remove trailing whitespace
kai["Stigma"] = kai["Stigma"].str.strip()

print("Unique Kai labels:", sorted(kai["Kai"].unique()))
print(f"Coverage: {kai['Stigma'].isin(intersectional_bias['stigma1']).sum()} / {len(kai)} stigmas matched")


In [ ]:
# Merge Kai labels for stigma1 and stigma2
ik = intersectional_bias.merge(
    kai.rename(columns={"Stigma": "stigma1", "Kai": "kai_A"}), on="stigma1", how="left"
)
ik = ik.merge(
    kai.rename(columns={"Stigma": "stigma2", "Kai": "kai_B"}), on="stigma2", how="left"
)

ik["kai_pair"] = ik.apply(
    lambda r: " × ".join(sorted([str(r["kai_A"]), str(r["kai_B"])])), axis=1
)

print(f"Unmatched stigma1: {ik['kai_A'].isna().sum()}")
print(f"Unmatched stigma2: {ik['kai_B'].isna().sum()}")
ik[["stigma1", "stigma2", "kai_A", "kai_B", "kai_pair", "interaction_effect"]].head(10)


In [ ]:
# Mean interaction effect by Kai label pair
kai_pair_summary = (
    ik.groupby("kai_pair")["interaction_effect"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_interaction", "std": "std_interaction"})
    .sort_values("mean_interaction", ascending=False)
    .round(3)
)
kai_pair_summary


In [ ]:
# Mean interaction effect by individual Kai label
kai_A_summary = ik.groupby("kai_A")["interaction_effect"].mean().rename("mean_interaction_as_A")
kai_B_summary = ik.groupby("kai_B")["interaction_effect"].mean().rename("mean_interaction_as_B")

kai_view = pd.concat([kai_A_summary, kai_B_summary], axis=1)
kai_view["mean_overall"] = kai_view.mean(axis=1)
kai_view.sort_values("mean_overall", ascending=False).round(3)


In [ ]:
# Bootstrap CIs — by Kai pair
kai_boot_rows = []
for pair, group in ik.groupby("kai_pair"):
    mean, lower, upper = bootstrap_ci(group["interaction_effect"].values)
    kai_boot_rows.append({
        "kai_pair":          pair,
        "mean_interaction":  round(mean, 4),
        "ci_lower":          round(lower, 4),
        "ci_upper":          round(upper, 4),
        "n_stigma_pairs":    len(group),
        "excludes_zero":     lower > 0 or upper < 0,
    })

kai_boot_pairs = pd.DataFrame(kai_boot_rows).sort_values("mean_interaction", ascending=False)
kai_boot_pairs


In [ ]:
# Bootstrap CIs — by individual Kai label
kai_boot_ind_rows = []
for label, group in ik.groupby("kai_A"):
    mean, lower, upper = bootstrap_ci(group["interaction_effect"].values)
    kai_boot_ind_rows.append({
        "kai_label":         label,
        "mean_interaction":  round(mean, 4),
        "ci_lower":          round(lower, 4),
        "ci_upper":          round(upper, 4),
        "n_pairs":           len(group),
        "excludes_zero":     lower > 0 or upper < 0,
    })

kai_boot_ind = pd.DataFrame(kai_boot_ind_rows).sort_values("mean_interaction", ascending=False)
kai_boot_ind
1

## 16. Multi-Model Comparison

Loads all available model result CSVs and computes interaction effects per model. Compares overall bias rates, cluster-pair interaction effects, and bootstrap CIs side by side.

In [220]:
import glob

clusters = pd.read_csv(DATA / 'templates/neostigmas.csv', usecols=['Stigma', 'Cluster'])

def compute_model_stats(csv_path):
    d = pd.read_csv(csv_path)
    d = d.drop(columns=['prompt'], errors='ignore')
    d['biased'] = (d['model_answer'] == d['biased_answer']).astype(int)
    v = d[d['model_answer'].isin(['yes', 'no'])].copy()

    single_b = v[v['stigma2'].isna()].groupby('stigma1')['biased'].mean().rename('single_bias_rate')

    raw = (
        v[v['stigma2'].notna()]
        .groupby(['stigma1', 'stigma2'])['biased'].mean()
        .reset_index().rename(columns={'biased': 'combined_bias_rate'})
    )
    raw['pair'] = raw.apply(lambda r: tuple(sorted([r['stigma1'], r['stigma2']])), axis=1)
    ib = raw.groupby('pair')['combined_bias_rate'].mean().reset_index()
    ib[['stigma1', 'stigma2']] = pd.DataFrame(ib['pair'].tolist(), index=ib.index)
    ib = ib.drop(columns='pair')
    ib = ib.merge(single_b.rename('bias_A'), left_on='stigma1', right_index=True, how='left')
    ib = ib.merge(single_b.rename('bias_B'), left_on='stigma2', right_index=True, how='left')
    ib['interaction_effect'] = ib['combined_bias_rate'] - (ib['bias_A'] + ib['bias_B']) / 2

    ib = ib.merge(clusters.rename(columns={'Stigma': 'stigma1', 'Cluster': 'cluster_A'}), on='stigma1', how='left')
    ib = ib.merge(clusters.rename(columns={'Stigma': 'stigma2', 'Cluster': 'cluster_B'}), on='stigma2', how='left')
    ib['cluster_pair'] = ib.apply(lambda r: ' × '.join(sorted([r['cluster_A'], r['cluster_B']])), axis=1)

    model_name = d['model'].iloc[0]
    overall_bias = v['biased'].mean()
    single_bias = v[v['stigma2'].isna()]['biased'].mean()
    inter_bias = v[v['stigma2'].notna()]['biased'].mean()
    return model_name, overall_bias, single_bias, inter_bias, ib

csvs = sorted(glob.glob(str(DATA / 'results__*__with_single__*.csv')))
print(f'Found {len(csvs)} model result files:')
for p in csvs:
    print(f'  {p}')

model_stats = {}
for csv_path in csvs:
    model_name, overall, single, inter, ib = compute_model_stats(csv_path)
    model_stats[model_name] = {'overall': overall, 'single': single, 'intersectional': inter, 'ib': ib}
    print(f'{model_name}: overall={overall:.3f}  single={single:.3f}  intersectional={inter:.3f}')


Found 2 model result files:
  ../data/results__granite__with_single__original_positive_doubt_base.csv
  ../data/results__mistral__with_single__original_positive_doubt_base.csv
granite: overall=0.334  single=0.275  intersectional=0.335
mistral: overall=0.310  single=0.267  intersectional=0.310


In [221]:
# Overall / single / intersectional bias rates per model
rate_rows = []
for m, s in model_stats.items():
    rate_rows.append({'model': m, 'overall_bias': round(s['overall'], 3),
                      'single_bias': round(s['single'], 3),
                      'intersectional_bias': round(s['intersectional'], 3)})

rates_df = pd.DataFrame(rate_rows).set_index('model')
rates_df['amplification'] = (rates_df['intersectional_bias'] - rates_df['single_bias']).round(3)
rates_df


,overall_bias,single_bias,intersectional_bias,amplification
model,,,,
granite,0.334,0.275,0.335,0.060
mistral,0.310,0.267,0.310,0.043


In [222]:
# Mean interaction effect by cluster pair — one column per model
cluster_frames = []
for m, s in model_stats.items():
    cp = (
        s['ib'].groupby('cluster_pair')['interaction_effect']
        .mean()
        .rename(m)
        .round(4)
    )
    cluster_frames.append(cp)

cluster_compare = pd.concat(cluster_frames, axis=1)
cluster_compare['mean_across_models'] = cluster_compare.mean(axis=1).round(4)
cluster_compare.sort_values('mean_across_models', ascending=False)


,granite,mistral,mean_across_models
cluster_pair,,,
2 - Threatening × 3 - Sociodemographic,0.1675,0.1428,0.1552
1 - Awkward × 2 - Threatening,0.1519,0.1189,0.1354
2 - Threatening × 4 - Innocuous Persistent,0.1363,0.0848,0.1106
2 - Threatening × 5 - Unappealing Persistent,0.1161,0.0535,0.0848
2 - Threatening × 2 - Threatening,0.0627,0.0492,0.0560
1 - Awkward × 5 - Unappealing Persistent,0.0556,0.0341,0.0448
5 - Unappealing Persistent × 5 - Unappealing Persistent,0.0765,0.0064,0.0414
3 - Sociodemographic × 5 - Unappealing Persistent,0.0380,0.0443,0.0412
4 - Innocuous Persistent × 5 - Unappealing Persistent,0.0571,0.0245,0.0408


In [223]:
# Bootstrap CIs per model per cluster pair
boot_model_rows = []
for m, s in model_stats.items():
    for pair, group in s['ib'].groupby('cluster_pair'):
        mean, lower, upper = bootstrap_ci(group['interaction_effect'].values)
        boot_model_rows.append({
            'model': m,
            'cluster_pair': pair,
            'mean_interaction': round(mean, 4),
            'ci_lower': round(lower, 4),
            'ci_upper': round(upper, 4),
            'excludes_zero': lower > 0 or upper < 0,
        })

boot_model_df = pd.DataFrame(boot_model_rows)

# Pivot to wide format: rows = cluster_pair, columns = model mean
pivot = boot_model_df.pivot(index='cluster_pair', columns='model', values='mean_interaction')
pivot['mean_across_models'] = pivot.mean(axis=1).round(4)
pivot.sort_values('mean_across_models', ascending=False).round(4)


model,granite,mistral,mean_across_models
cluster_pair,,,
2 - Threatening × 3 - Sociodemographic,0.1675,0.1428,0.1552
1 - Awkward × 2 - Threatening,0.1519,0.1189,0.1354
2 - Threatening × 4 - Innocuous Persistent,0.1363,0.0848,0.1106
2 - Threatening × 5 - Unappealing Persistent,0.1161,0.0535,0.0848
2 - Threatening × 2 - Threatening,0.0627,0.0492,0.0560
1 - Awkward × 5 - Unappealing Persistent,0.0556,0.0341,0.0448
5 - Unappealing Persistent × 5 - Unappealing Persistent,0.0765,0.0064,0.0414
3 - Sociodemographic × 5 - Unappealing Persistent,0.0380,0.0443,0.0412
4 - Innocuous Persistent × 5 - Unappealing Persistent,0.0571,0.0245,0.0408


In [224]:
# Distribution of interaction effects per model — mean, median, % amplifying
dist_rows = []
for m, s in model_stats.items():
    ie = s['ib']['interaction_effect'].dropna()
    dist_rows.append({
        'model': m,
        'mean_IE': round(ie.mean(), 4),
        'median_IE': round(ie.median(), 4),
        'pct_amplifying': round(100 * (ie > 0).mean(), 1),
        'pct_dampening': round(100 * (ie < 0).mean(), 1),
        'n_pairs': len(ie),
    })

pd.DataFrame(dist_rows).set_index('model')


,mean_IE,median_IE,pct_amplifying,pct_dampening,n_pairs
model,,,,,
granite,0.060,0.0535,76.5,23.3,6216
mistral,0.042,0.0352,80.2,19.3,6216
